# Projet Réseaux de Neurones — Chargement et Préparation des Données
### Données Sentinel-2 — Arkansas & California 2021

## 1. Imports

In [2]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

## 2. Paramètres globaux

In [3]:
dossier = r"C:\Users\ibm\OneDrive\Documents\Projet Res Neur\data"

NB_DATES    = 36
NB_BANDES   = 10
NB_SAMPLES  = 10000
TRAIN       = 240
VAL         = 60
RANDOM_SEED = 42

CODES_NON_AGRICOLES = set(range(111, 200))

## 3. Mappers de classes

In [4]:
def mapper_arkansas(c):
    if c == 1:                     return 'Corn'
    elif c == 2:                   return 'Cotton'
    elif c == 3:                   return 'Rice'
    elif c == 5:                   return 'Soybeans'
    elif c in CODES_NON_AGRICOLES: return None
    elif c > 0:                    return 'Others'
    else:                          return None

def mapper_california(c):
    if c == 69:                    return 'Grapes'
    elif c == 36:                  return 'Alfalfa'
    elif c == 3:                   return 'Rice'
    elif c == 75:                  return 'Almonds'
    elif c == 204:                 return 'Pistachios'
    elif c in CODES_NON_AGRICOLES: return None
    elif c > 0:                    return 'Others'
    else:                          return None

## 4. Fonction de chargement d'une zone

In [5]:
def charger_zone(nom_sentinel, nom_cdl, mapper):
    print(f"  Lecture {nom_sentinel}...")
    with rasterio.open(dossier + f"\\{nom_sentinel}") as src:
        sentinel  = src.read().astype(np.float32)
        transform = src.transform
        crs       = src.crs
    H, W = sentinel.shape[1], sentinel.shape[2]
    with rasterio.open(dossier + f"\\{nom_cdl}") as src:
        cdl = np.zeros((H, W), dtype=np.uint8)
        reproject(source=rasterio.band(src, 1),
                  destination=cdl,
                  dst_transform=transform,
                  dst_crs=crs,
                  resampling=Resampling.nearest)
    labels_flat   = np.array([mapper(int(c)) for c in cdl.flatten()], dtype=object)
    sentinel_flat = sentinel.reshape(NB_DATES, NB_BANDES, H * W).transpose(2, 0, 1)
    masque_flat   = labels_flat != None
    print(f"  -> {int(masque_flat.sum()):,} pixels agricoles valides sur {H*W:,} total")
    return sentinel_flat, labels_flat, masque_flat

## 5. Fonction de sampling sur deux zones

In [6]:
def sampler_deux_zones(classes, n_total, choix_zone,
                       s_nord, labels_nord, masque_nord,
                       s_sud,  labels_sud,  masque_sud,
                       seed, classes_forcees=None):

    if classes_forcees is None:
        classes_forcees = []

    n_nord = {c: int(((labels_nord == c) & masque_nord).sum()) for c in classes}
    n_sud  = {c: int(((labels_sud  == c) & masque_sud ).sum()) for c in classes}

    print(f"\n  {'Culture':<14} {'Nord':>10} {'Sud':>10} {'-> Zone':>10}")
    print(f"  {'-'*48}")
    for c in classes:
        print(f"  {c:<14} {n_nord[c]:>10,} {n_sud[c]:>10,} {choix_zone[c]:>10}")

    dispo       = {c: n_nord[c] if choix_zone[c] == 'Nord' else n_sud[c] for c in classes}
    total_dispo = sum(dispo.values())
    nb          = {c: round(n_total * dispo[c] / total_dispo) for c in classes}

    # Ajustement pour tomber exactement sur n_total
    diff = n_total - sum(nb.values())
    nb['Others'] += diff

    # Classes < 5% -> Others SAUF classes_forcees
    seuil_5pct = 0.05 * n_total
    for c in list(nb.keys()):
        if c != 'Others' and c not in classes_forcees and nb[c] < seuil_5pct:
            print(f"  -> {c} ({nb[c]} px = {100*nb[c]/n_total:.1f}%) fusionne dans Others")
            nb['Others'] += nb.pop(c)

    # Garantir minimum 500 samples pour les classes forcées
    MIN_FORCE = 500
    for c in classes_forcees:
        if c in nb and nb[c] < MIN_FORCE:
            manque = MIN_FORCE - nb[c]
            nb[c]  = MIN_FORCE
            autres = [x for x in nb if x != c and nb[x] > MIN_FORCE]
            tot    = sum(nb[x] for x in autres)
            for x in autres:
                nb[x] -= round(manque * nb[x] / tot)
            diff2 = n_total - sum(nb.values())
            nb[max(nb, key=lambda c: nb[c])] += diff2
            print(f"  -> {c} forcé à {MIN_FORCE} samples minimum")

    print(f"\n  {'Culture':<14} {'Dispo':>10} {'-> Samples':>12} {'Zone':>8}")
    print(f"  {'-'*48}")
    for c in classes:
        if c in nb:
            flag = " [FORCÉ]" if c in classes_forcees else ""
            print(f"  {c:<14} {dispo[c]:>10,} {nb[c]:>12,} "
                  f"{choix_zone.get(c,'?'):>8}{flag}")
    print(f"  {'TOTAL':<14} {total_dispo:>10,} {sum(nb.values()):>12,}")

    np.random.seed(seed)
    tous_pixels, tous_labels = [], []

    for classe, n_voulu in nb.items():
        zone     = choix_zone.get(classe, 'Nord')
        s_z      = s_nord      if zone == 'Nord' else s_sud
        labels_z = labels_nord if zone == 'Nord' else labels_sud
        qmask_z  = masque_nord if zone == 'Nord' else masque_sud

        idx = np.where((labels_z == classe) & qmask_z)[0]
        if len(idx) < n_voulu:
            print(f"  ATTENTION {classe}: dispo={len(idx):,} < voulu={n_voulu:,} -> remise")
            sampled = np.random.choice(idx, n_voulu, replace=True)
        else:
            sampled = np.random.choice(idx, n_voulu, replace=False)

        tous_pixels.append(s_z[sampled])
        tous_labels.extend([classe] * n_voulu)
        print(f"  OK {classe:<14} ({zone:4}) : {n_voulu:,} pixels")

    return np.concatenate(tous_pixels), np.array(tous_labels)

## 6. Chargement Arkansas

In [7]:
print("\n=== ARKANSAS ===")

s_ar_nord, labels_ar_nord, masque_ar_nord = charger_zone(
    "sentinel2_arkansas_north_2021.tif",
    "cdl_arkansas_north_2021.tif",
    mapper_arkansas
)
s_ar_sud, labels_ar_sud, masque_ar_sud = charger_zone(
    "sentinel2_arkansas_south_2021.tif",
    "cdl_arkansas_south_2021.tif",
    mapper_arkansas
)


=== ARKANSAS ===
  Lecture sentinel2_arkansas_north_2021.tif...
  -> 1,801,754 pixels agricoles valides sur 2,758,016 total
  Lecture sentinel2_arkansas_south_2021.tif...
  -> 1,834,616 pixels agricoles valides sur 2,759,502 total


In [ ]:
classes_ar    = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']
choix_zone_ar = {
    'Soybeans': 'Nord',
    'Rice'    : 'Nord',
    'Corn'    : 'Sud',
    'Cotton'  : 'Sud',
    'Others'  : 'Nord',
}

pixels_ar, labels_ar = sampler_deux_zones(
    classes_ar, NB_SAMPLES, choix_zone_ar,
    s_ar_nord, labels_ar_nord, masque_ar_nord,
    s_ar_sud,  labels_ar_sud,  masque_ar_sud,
    RANDOM_SEED
)

## 7. Chargement California

In [9]:
print("\n=== CALIFORNIA ===")

s_cal_nord, labels_cal_nord, masque_cal_nord = charger_zone(
    "sentinel2_california_north_2021.tif",
    "cdl_california_north_2021.tif",
    mapper_california
)
s_cal_sud, labels_cal_sud, masque_cal_sud = charger_zone(
    "sentinel2_california_south_2021.tif",
    "cdl_california_south_2021.tif",
    mapper_california
)


=== CALIFORNIA ===
  Lecture sentinel2_california_north_2021.tif...
  -> 700,847 pixels agricoles valides sur 1,034,906 total
  Lecture sentinel2_california_south_2021.tif...
  -> 804,121 pixels agricoles valides sur 1,035,835 total


In [ ]:
classes_cal = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

n_nord_cal = {c: int(((labels_cal_nord == c) & masque_cal_nord).sum()) for c in classes_cal}
n_sud_cal  = {c: int(((labels_cal_sud  == c) & masque_cal_sud ).sum()) for c in classes_cal}
choix_zone_cal = {
    c: 'Nord' if n_nord_cal[c] >= n_sud_cal[c] else 'Sud'
    for c in classes_cal
}

pixels_cal, labels_cal = sampler_deux_zones(
    classes_cal, NB_SAMPLES, choix_zone_cal,
    s_cal_nord, labels_cal_nord, masque_cal_nord,
    s_cal_sud,  labels_cal_sud,  masque_cal_sud,
    RANDOM_SEED,
    classes_forcees=['Alfalfa', 'Pistachios']
)

## 8. Sauvegarde des fichiers .npy

In [11]:
np.save(dossier + r"\pixels_arkansas.npy",   pixels_ar)
np.save(dossier + r"\labels_arkansas.npy",   labels_ar)
np.save(dossier + r"\pixels_california.npy", pixels_cal)
np.save(dossier + r"\labels_california.npy", labels_cal)
print("\nFichiers .npy sauvegardes.")


Fichiers .npy sauvegardes.


## 9. Table 2 — Répartition des samples

In [12]:
def afficher_table2(labels, classes, zone_nom):
    total_s = total_tr = total_v = total_te = 0
    premiere = True
    for classe in classes:
        n = int((labels == classe).sum())
        if n == 0:
            continue
        tr = TRAIN; v = VAL; te = max(0, n - tr - v)
        total_s += n; total_tr += tr; total_v += v; total_te += te
        z = zone_nom if premiere else ""
        premiere = False
        print(f"  {z:<12} {classe:<14} {n:>10,} {tr:>10,} {v:>12,} {te:>10,}")
    print(f"  {'':12} {'All':<14} {total_s:>10,} {total_tr:>10,} {total_v:>12,} {total_te:>10,}")

print("\n")
print("=" * 74)
print("  Table 2 -- Numbers of the samples in the study areas")
print("=" * 74)
print(f"  {'Study area':<12} {'Class Name':<14} {'Samples':>10} "
      f"{'Training':>10} {'Validation':>12} {'Testing':>10}")
print("-" * 74)
afficher_table2(labels_ar,  classes_ar,  "Arkansas")
print("-" * 74)
afficher_table2(labels_cal, classes_cal, "California")
print("=" * 74)

print(f"\nTermine !")
print(f"   pixels_ar  : {pixels_ar.shape}")
print(f"   labels_ar  : {labels_ar.shape}")
print(f"   pixels_cal : {pixels_cal.shape}")
print(f"   labels_cal : {labels_cal.shape}")



  Table 2 -- Numbers of the samples in the study areas
  Study area   Class Name        Samples   Training   Validation    Testing
--------------------------------------------------------------------------
  Arkansas     Soybeans            4,272        240           60      3,972
               Rice                2,523        240           60      2,223
               Corn                1,238        240           60        938
               Cotton              1,201        240           60        901
               Others                766        240           60        466
               All                10,000      1,200          300      8,500
--------------------------------------------------------------------------
  California   Grapes              2,044        240           60      1,744
               Rice                1,370        240           60      1,070
               Alfalfa               500        240           60        200
               Almonds           